# مدار — مقارنة إزالة الموسيقى مع الحفاظ على الحوار والمؤثرات

**للتجربة قبل اعتماد نموذج جديد في التطبيق.** اختر من القائمة **Runtime → Change runtime type → T4 GPU**، ثم **Run all**.
ارفع مقطعًا فيه الموسيقى التي لم يزلها النموذج الحالي. الافتراضي اختبار أول 30 ثانية؛ يمكنك تغيير البداية والمدة قبل التشغيل.

ستحصل على مقارنة لنفس الجزء:
1. الصوت الأصلي.
2. النموذج الحالي نفسه في التطبيق: **Inst HQ4 ثم HQ3**.
3. المرشح **Bandit v2 Multi**: فصل الحوار والموسيقى والمؤثرات، ثم جمع **الحوار + المؤثرات** فقط.

المرشح متخصص في الفصل السينمائي لكنه ليس ضمانًا لإزالة كل الموسيقى أو الحفاظ على كل مؤثر. احكم بالسماع على الكلام والمؤثرات وبقايا الموسيقى، ولا تعتمد على ارتفاع أو انخفاض الصوت وحده.
لا تتغير إعدادات تطبيقك ولا تُرسل نتيجة جديدة إليه. تُرفع ملفات التجربة إلى جلسة Colab بحسابك، ولا يلزم ربط Drive أو حساب إضافي.

المراجع: [البحث وكود Bandit v2](https://github.com/kwatcharasupat/bandit-v2)، [الأوزان الأصلية](https://zenodo.org/records/12701995)، [حزمة التشغيل](https://github.com/openmirlab/bandit-infer).
الكود Apache-2.0؛ أوزان Bandit v2 بترخيص CC-BY-SA-4.0. سيبقى دمجه في التطبيق خطوة منفصلة بعد التجربة.


In [ ]:
#@title 1. اختيار طريقة الفصل
PIPELINE = "inst_hq_4_3"
START_SECONDS = 0 #@param {type:"number"}
DURATION_SECONDS = 30 #@param {type:"number"}
SECOND_PASS = False #@param {type:"boolean"}
# المرحلة الإضافية اختيارية: قد تقلل بقايا الموسيقى لكنها قد تضعف المؤثرات أيضًا.
if START_SECONDS < 0 or not 8 <= DURATION_SECONDS <= 300:
    raise ValueError("اختر بداية من الصفر فأعلى ومدة بين 8 و300 ثانية")
import json, os, sys, subprocess, time, uuid, hashlib, zlib, wave
from pathlib import Path
subprocess.run(["nvidia-smi", "-L"], check=True)
SERVICE_ID = "madar_" + uuid.uuid4().hex
ROOT = Path("/content") / SERVICE_ID
ROOT.mkdir()
JOB = ROOT.parent / (SERVICE_ID + "_b0")
JOB.mkdir()
RESULTS = ROOT.parent / ("madar_results_" + uuid.uuid4().hex)
RESULTS.mkdir()


In [ ]:
#@title 2. كود Colab الكامل — مطابق لمصادر التطبيق
SOURCES = {'music_worker.py': '"""UVR Voc_FT with unchanged audio windows; one CUDA model per viewing session."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport os\nimport subprocess\nimport sys\nimport urllib.request\nimport time\nimport shutil\nimport re\nimport zlib\nimport threading\n\n_STATUS_LOCK = threading.RLock()\n_STATUS_PATH = None\n_STATUS = {}\n_HEARTBEAT = 0\n_PROTOCOL = 1\n_SERVICE_ROOT = None\n_SERVICE_ERROR = None\n_ACTIVE_JOB = ""\n_NEXT_BATCH = 0\n_FX_CLIENT = None\n_FX_VERIFIED = False\nFX_PIPELINE = \'inst_hq_4_3\'\nFX_SHA = \'e165729735b0493b2711b73de37f17cd57dc0a550bd2dff1d4c627586ed32a55\'\n\ndef load_private_cudnn():\n    global _cudnn\n    if os.environ.get(\'MADAR_CUDNN_LIB\'):\n        import ctypes\n        _cudnn = ctypes.CDLL(os.environ[\'MADAR_CUDNN_LIB\'], mode=ctypes.RTLD_GLOBAL)\n        _cudnn.cudnnGetVersion.restype = ctypes.c_size_t\n        if _cudnn.cudnnGetVersion() != 91300:\n            raise ValueError(\'MUSIC_CUDNN_VERSION\')\n\nENGINE = None\nGPU_FREE_BEFORE = None\nGPU_VERIFIED = False\nCACHE = None\n\nRATE = 44100\nCORE = 4 * RATE\nMODEL_SHA = \'e411182ce2c53541fefdcc99a8f46f2fe03978eb22038b9497c1d9d95a00fad4\'\nMODEL_URL = \'https://github.com/k2-fsa/sherpa-onnx/releases/download/source-separation-models/UVR-MDX-NET-Voc_FT.onnx\'\n\n\ndef write_json(path, value):\n    p = Path(path)\n    tmp = p.with_suffix(\'.tmp\')\n    tmp.write_text(json.dumps(value), encoding=\'utf-8\')\n    tmp.replace(p)\n\n\ndef status(stage, progress, parts=None, frames=None, packs=None, extra=None):\n    global _STATUS_PATH, _STATUS, _HEARTBEAT\n    value = dict(stage=stage, progress=progress)\n    if _PROTOCOL == 2:\n        value.update(job=Path.cwd().name, protocol=2)\n    if extra is not None:\n        value.update(extra)\n    if parts is not None:\n        value.update(chunks=parts, frames=frames, sample_rate=RATE, channels=2, model_sha256=MODEL_SHA)\n    if packs is not None:\n        value[\'packs\'] = packs\n    with _STATUS_LOCK:\n        _HEARTBEAT += 1\n        value[\'heartbeat\'] = _HEARTBEAT\n        _STATUS_PATH = Path(\'progress.json\').resolve()\n        _STATUS = value\n        write_json(_STATUS_PATH, value)\n\n\ndef service_state():\n    if _SERVICE_ROOT is not None:\n        value = dict(protocol=2, heartbeat=_HEARTBEAT, next_batch=_NEXT_BATCH,\n                     stage=\'failed\' if _SERVICE_ERROR else \'busy\' if _ACTIVE_JOB else \'ready\' if GPU_VERIFIED or _FX_VERIFIED else \'bootstrap\',\n                     active_job=_ACTIVE_JOB)\n        if _SERVICE_ERROR:\n            value[\'error\'] = _SERVICE_ERROR\n        write_json(_SERVICE_ROOT / \'service.json\', value)\n\n\ndef heartbeat(stop, interval=2):\n    global _HEARTBEAT\n    while not stop.wait(interval):\n        if _SERVICE_ROOT is not None:\n            try:\n                request = json.loads((_SERVICE_ROOT / \'stop.json\').read_text(\'utf-8\'))\n            except (OSError, ValueError):\n                request = {}\n            if request == dict(service=_SERVICE_ROOT.name, cancel=True):\n                # Only this owned worker process exits, even while inference is busy.\n                write_json(_SERVICE_ROOT / \'stopped.json\', dict(service=_SERVICE_ROOT.name, stopped=True))\n                os._exit(0)\n        with _STATUS_LOCK:\n            if _STATUS_PATH is not None and _STATUS.get(\'stage\') not in (\'failed\', \'done\'):\n                _HEARTBEAT += 1\n                _STATUS[\'heartbeat\'] = _HEARTBEAT\n                write_json(_STATUS_PATH, _STATUS)\n            if _SERVICE_ROOT is not None:\n                _HEARTBEAT += 1\n                service_state()\n\n\ndef report_failure(error):\n    global _SERVICE_ERROR\n    code = str(error) if isinstance(error, ValueError) and re.fullmatch(r\'[A-Z][A-Z0-9_]{0,79}\', str(error)) else type(error).__name__\n    if isinstance(error, RuntimeError) and (\'CUDNN Frontend\' in str(error) or \'HEURISTIC_QUERY_FAILED\' in str(error)):\n        code = \'MUSIC_CUDNN_CONV\'\n    with _STATUS_LOCK:\n        previous = _STATUS.get(\'stage\', \'bootstrap\')\n        path = _STATUS_PATH or Path(\'progress.json\').resolve()\n        value = dict(stage=\'failed\', last_stage=previous, progress=0, error=code)\n        if _PROTOCOL == 2:\n            value[\'job\'] = Path.cwd().name\n        write_json(path, value)\n        _SERVICE_ERROR = code\n        service_state()\n\n\ndef validate_request(req):\n    frames = req.get(\'frames\')\n    chunks = req.get(\'chunks\')\n    if type(frames) is not int or not 0 < frames <= 600 * RATE:\n        raise ValueError(\'MUSIC_DURATION\')\n    if not isinstance(chunks, list) or len(chunks) != (frames + CORE - 1) // CORE:\n        raise ValueError(\'MUSIC_CHUNKS\')\n    for i, part in enumerate(chunks):\n        if (not isinstance(part, dict) or part.get(\'bytes\') != min(CORE, frames - i * CORE) * 4\n                or not isinstance(part.get(\'sha256\'), str) or len(part[\'sha256\']) != 64):\n            raise ValueError(\'MUSIC_MANIFEST\')\n    return frames, chunks\n\n\ndef prepare():\n    global ENGINE, GPU_FREE_BEFORE, GPU_VERIFIED\n    if ENGINE is None:\n        status(\'gpu_check\', 27)\n        load_private_cudnn()\n        import torch\n        if not torch.cuda.is_available() or not torch.version.cuda:\n            raise ValueError(\'MUSIC_GPU_UNAVAILABLE\')\n        major = int(torch.version.cuda.split(\'.\')[0])\n        cuda = 13 if major >= 13 else 12\n        version = f\'1.13.8+cuda{cuda}.cudnn9.onnxruntime1.28.2\'\n        status(\'install\', 28)\n        from importlib.metadata import PackageNotFoundError, version as installed_version\n        try:\n            installed = installed_version(\'sherpa-onnx\') == version and installed_version(\'numpy\') == \'2.2.6\'\n        except PackageNotFoundError:\n            installed = False\n        if not installed:\n            with open(\'install.log\', \'wb\') as log:\n                subprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'--disable-pip-version-check\',\n                                \'--only-binary=:all:\', \'--timeout\', \'20\', \'--retries\', \'2\',\n                                f\'sherpa-onnx=={version}\', \'numpy==2.2.6\',\n                                \'-f\', \'https://k2-fsa.github.io/sherpa/onnx/cuda.html\'],\n                               stdout=log, stderr=log, check=True, timeout=600)\n        # Importing torch first loads the CUDA/cuDNN libraries used by this runtime.\n        import sherpa_onnx\n        from importlib.metadata import version as installed_version\n        if version != installed_version(\'sherpa-onnx\'):\n            raise ValueError(\'MUSIC_GPU_WHEEL\')\n        status(\'model_download\', 32)\n        model = (CACHE or Path.cwd()) / \'vocals.onnx\'\n        if not model.is_file() or hashlib.sha256(model.read_bytes()).hexdigest() != MODEL_SHA:\n            temporary = model.with_suffix(\'.download\')\n            digest = hashlib.sha256()\n            with urllib.request.urlopen(MODEL_URL, timeout=60) as response, temporary.open(\'wb\') as out:\n                size = 0\n                while True:\n                    data = response.read(1024 * 1024)\n                    if not data:\n                        break\n                    size += len(data)\n                    if size > 100 * 1024 * 1024:\n                        raise ValueError(\'MODEL_SIZE\')\n                    digest.update(data)\n                    out.write(data)\n            if digest.hexdigest() != MODEL_SHA:\n                temporary.unlink(missing_ok=True)\n                raise ValueError(\'MODEL_HASH\')\n            temporary.replace(model)\n        status(\'model_load\', 38)\n        GPU_FREE_BEFORE = torch.cuda.mem_get_info()[0]\n        config = sherpa_onnx.OfflineSourceSeparationConfig(\n            model=sherpa_onnx.OfflineSourceSeparationModelConfig(\n                uvr=sherpa_onnx.OfflineSourceSeparationUvrModelConfig(model=str(model)),\n                num_threads=2, provider=\'cuda\', debug=False))\n        if not config.validate():\n            raise ValueError(\'MODEL_CONFIG\')\n        ENGINE = sherpa_onnx.OfflineSourceSeparation(config)\n        verify_gpu_inference()\n        status(\'ready\', 38)\n\n\ndef prepare_fx():\n    global _FX_CLIENT, _FX_VERIFIED\n    if _FX_CLIENT is None:\n        from music_fx_bridge import Client\n        _FX_CLIENT = Client(CACHE or Path.cwd(), status)\n    _FX_CLIENT.prepare()\n    _FX_VERIFIED = True\n    status(\'ready\', 38)\n\n\ndef publish_fx(frames):\n    output, gpu_ms = _FX_CLIENT.run(Path.cwd(), frames)\n    stride, recent, count = 16 * RATE, [], 0\n    try:\n        with output.open(\'rb\') as stream:\n            while True:\n                data = stream.read(stride * 4)\n                if not data: break\n                meta = write_pack(f\'pack_{count}.pcm\', data)\n                meta[\'index\'] = count\n                write_json(f\'pack_{count}.json\', meta)\n                recent.append(meta); recent = recent[-8:]; count += 1\n                complete = min(frames, count * stride)\n                status(\'separate\', 90 + int(7 * complete / frames), extra=dict(\n                    frames=frames, completed_frames=complete, sample_rate=RATE, channels=2,\n                    model_sha256=FX_SHA, pipeline=FX_PIPELINE, pack_count=count,\n                    output_frames=stride, recent_packs=recent, gpu_ms=gpu_ms))\n        manifest = dict(protocol=2, job=Path.cwd().name, frames=frames, completed_frames=frames,\n            sample_rate=RATE, channels=2, model_sha256=FX_SHA, pipeline=FX_PIPELINE,\n            provider=\'cuda\', pack_count=count, output_frames=stride,\n            recent_packs=recent, gpu_ms=gpu_ms)\n        write_json(\'result.json\', manifest)\n        status(\'done\', 97, extra=manifest)\n    finally:\n        output.unlink(missing_ok=True)\n        Path(\'input.pcm\').unlink(missing_ok=True)\n\n\ndef verify_gpu_inference():\n    global GPU_VERIFIED\n    status(\'gpu_inference_check\', 38)\n    import numpy as np\n    import torch\n    # Same maximum context length as a real core. Never publish this synthetic audio.\n    samples = np.zeros((2, 6 * RATE), dtype=np.float32)\n    output = ENGINE.process(sample_rate=RATE, samples=samples)\n    if output.sample_rate != RATE or len(output.stems) != 2:\n        raise ValueError(\'MUSIC_OUTPUT_FORMAT\')\n    vocals = output.stems[0].data\n    if vocals.ndim != 2 or vocals.shape[0] != 2 or vocals.shape[1] < samples.shape[1] or not np.isfinite(vocals).all():\n        raise ValueError(\'MUSIC_OUTPUT_LENGTH\')\n    torch.cuda.synchronize()\n    if GPU_FREE_BEFORE - torch.cuda.mem_get_info()[0] < 8 * 1024 * 1024:\n        raise ValueError(\'MUSIC_GPU_PROVIDER\')\n    GPU_VERIFIED = True\n\n\ndef read_wire(part, name, expected):\n    encoding = part.get(\'wire_encoding\', \'pcm\')\n    if encoding not in (\'pcm\', \'zlib\'):\n        raise ValueError(\'MUSIC_WIRE_ENCODING\')\n    size = part.get(\'wire_bytes\') if encoding == \'zlib\' else expected\n    if type(size) is not int or not 0 < size <= expected or (encoding == \'zlib\' and size >= expected):\n        raise ValueError(\'MUSIC_WIRE_LENGTH\')\n    path = Path(name + (\'.zlib\' if encoding == \'zlib\' else \'\'))\n    if path.stat().st_size != size:\n        raise ValueError(\'MUSIC_WIRE_LENGTH\')\n    data = path.read_bytes()\n    if encoding == \'zlib\':\n        if hashlib.sha256(data).hexdigest() != part.get(\'wire_sha256\'):\n            raise ValueError(\'MUSIC_WIRE_HASH\')\n        try:\n            engine = zlib.decompressobj()\n            data = engine.decompress(data, expected + 1)\n            if len(data) != expected or not engine.eof or engine.unconsumed_tail or engine.unused_data:\n                raise ValueError(\'MUSIC_WIRE_LENGTH\')\n        except zlib.error as error:\n            raise ValueError(\'MUSIC_WIRE_HASH\') from error\n    if len(data) != expected or hashlib.sha256(data).hexdigest() != part.get(\'sha256\'):\n        raise ValueError(\'UPLOAD_HASH\')\n    path.unlink()\n    return data\n\n\ndef decode_source(source, seconds=None):\n    """Decodes the clip\'s own compressed audio (sent in parts, a tenth of the decoded size) into input.pcm at\n    44.1 kHz stereo 16-bit, exactly what the decoded packs carried, and returns its length in frames."""\n    import subprocess\n    parts = source.get(\'parts\') if isinstance(source, dict) else None\n    if not isinstance(parts, list) or not 0 < len(parts) <= 128:\n        raise ValueError(\'MUSIC_SOURCE_MANIFEST\')\n    digest, total = hashlib.sha256(), 0\n    with open(\'source.m4a\', \'wb\') as out:\n        for i, part in enumerate(parts):\n            size = part.get(\'bytes\') if isinstance(part, dict) else None\n            if type(size) is not int or not 0 < size <= 4 * 1024 * 1024:\n                raise ValueError(\'MUSIC_SOURCE_MANIFEST\')\n            path = Path(f\'source_{i}.bin\')\n            data = path.read_bytes() if path.is_file() and path.stat().st_size == size else b\'\'\n            if len(data) != size or hashlib.sha256(data).hexdigest() != part.get(\'sha256\'):\n                raise ValueError(\'MUSIC_SOURCE_HASH\')\n            out.write(data)\n            digest.update(data)\n            total += size\n            path.unlink()\n    if total != source.get(\'bytes\') or digest.hexdigest() != source.get(\'sha256\'):\n        raise ValueError(\'MUSIC_SOURCE_HASH\')\n    probe = subprocess.run([\'ffprobe\', \'-v\', \'error\', \'-select_streams\', \'a:0\', \'-show_entries\', \'stream=channels,codec_name\',\n                            \'-of\', \'json\', \'source.m4a\'], capture_output=True, text=True, timeout=60)\n    try:\n        stream = json.loads(probe.stdout)[\'streams\'][0]\n        channels = int(stream[\'channels\'])\n    except (ValueError, KeyError, IndexError, TypeError) as error:\n        raise ValueError(\'MUSIC_SOURCE_UNREADABLE\') from error\n    if not 1 <= channels <= 8:\n        raise ValueError(\'MUSIC_SOURCE_UNREADABLE\')\n    command = [\'ffmpeg\', \'-v\', \'error\', \'-nostdin\', \'-y\', \'-i\', \'source.m4a\', \'-map\', \'0:a:0\', \'-vn\']\n    if type(seconds) in (int, float) and 0 < seconds <= 24 * 3600:\n        command += [\'-t\', f\'{seconds:.6f}\']\n    # The encoder\'s priming before the clip\'s time 0 is dropped to the sample, as the phone\'s decoder drops it\n    # (kept, the sound came 21 ms late against the picture). A mono clip fills both channels at full level, as the\n    # phone\'s own decoder did (ffmpeg\'s default lowers it 3 dB).\n    lead = source.get(\'lead_seconds\', 0)\n    if type(lead) not in (int, float) or not 0 <= lead <= 2:\n        raise ValueError(\'MUSIC_SOURCE_MANIFEST\')\n    trim = [f\'aresample={RATE}\', f\'atrim=start_sample={round(lead * RATE)}\', \'asetpts=PTS-STARTPTS\'] if lead > 0 else []\n    command += [\'-af\', \',\'.join(trim + ([\'pan=stereo|c0=c0|c1=c0\'] if channels == 1 else [\'aformat=channel_layouts=stereo\']))]\n    command += [\'-ar\', str(RATE), \'-f\', \'s16le\', \'-acodec\', \'pcm_s16le\', \'input.pcm\']\n    done = subprocess.run(command, capture_output=True, timeout=900)\n    Path(\'source.m4a\').unlink(missing_ok=True)\n    size = Path(\'input.pcm\').stat().st_size if Path(\'input.pcm\').is_file() else 0\n    if done.returncode != 0 or size < 4 or size % 4:\n        raise ValueError(\'MUSIC_SOURCE_DECODE\')\n    return size // 4\n\n\ndef write_pack(name, data):\n    # Keep raw files for legacy clients; announce compressed bytes only when smaller.\n    Path(name).write_bytes(data)\n    part = dict(bytes=len(data), sha256=hashlib.sha256(data).hexdigest())\n    encoded = zlib.compress(data, level=1)\n    if len(encoded) + 512 < len(data):\n        Path(name + \'.zlib\').write_bytes(encoded)\n        part.update(wire_encoding=\'zlib\', wire_bytes=len(encoded), wire_sha256=hashlib.sha256(encoded).hexdigest())\n    return part\n\n\ndef run():\n    global GPU_VERIFIED\n    req = json.loads(Path(\'request.json\').read_text(\'utf-8\'))\n    if req.get(\'protocol\') == 2:\n        return run_v2(req)\n    frames, chunks = validate_request(req)\n    prepare()\n    import numpy as np\n    inputs = req.get(\'inputs\', chunks)\n    stride = 4 * CORE if \'inputs\' in req else CORE\n    if not isinstance(inputs, list) or len(inputs) != (frames + stride - 1) // stride:\n        raise ValueError(\'MUSIC_MANIFEST\')\n    with open(\'input.pcm\', \'wb\') as out:\n        for i, part in enumerate(inputs):\n            if not isinstance(part, dict) or part.get(\'bytes\') != min(stride, frames-i*stride)*4:\n                raise ValueError(\'MUSIC_MANIFEST\')\n            data = read_wire(part, f\'input_{i}.pcm\', min(stride, frames-i*stride)*4)\n            for at in range(0, len(data), CORE*4):\n                index = (i*stride + at//4) // CORE\n                block = data[at:at+CORE*4]\n                if hashlib.sha256(block).hexdigest() != chunks[index][\'sha256\']:\n                    raise ValueError(\'UPLOAD_HASH\')\n            out.write(data)\n    source = np.memmap(\'input.pcm\', dtype=\'<i2\', mode=\'r\', shape=(frames, 2))\n    parts, packs, packed = [], [], bytearray()\n    for i, start in enumerate(range(0, frames, CORE)):\n        left, right = max(0, start - RATE), min(frames, start + CORE + RATE)\n        samples = np.ascontiguousarray(source[left:right].T.astype(np.float32) / 32768)\n        output = ENGINE.process(sample_rate=RATE, samples=samples)\n        if not GPU_VERIFIED:\n            import torch\n            torch.cuda.synchronize()\n            # Detect a native provider silently falling back to CPU. This model\'s\n            # GPU weights alone exceed 8 MiB; no CPU result is labelled GPU.\n            if GPU_FREE_BEFORE - torch.cuda.mem_get_info()[0] < 8 * 1024 * 1024:\n                raise ValueError(\'MUSIC_GPU_PROVIDER\')\n            GPU_VERIFIED = True\n        if output.sample_rate != RATE or len(output.stems) != 2:\n            raise ValueError(\'MUSIC_OUTPUT_FORMAT\')\n        vocals = output.stems[0].data\n        keep, offset = min(CORE, frames - start), start - left\n        if vocals.ndim != 2 or vocals.shape[0] != 2 or vocals.shape[1] < offset + keep:\n            raise ValueError(\'MUSIC_OUTPUT_LENGTH\')\n        values = vocals[:, offset:offset + keep].T\n        if not np.isfinite(values).all():\n            raise ValueError(\'MUSIC_OUTPUT_FINITE\')\n        # Java Math.round(x) is floor(x + .5), including negative samples.\n        pcm = np.floor(np.clip(values, -1, 1) * 32767 + .5).astype(\'<i2\').tobytes()\n        Path(f\'output_{i}.pcm\').write_bytes(pcm)\n        parts.append(dict(bytes=len(pcm), sha256=hashlib.sha256(pcm).hexdigest()))\n        packed.extend(pcm)\n        if (i + 1) % 4 == 0 or start + CORE >= frames:\n            data = bytes(packed)\n            packs.append(write_pack(f\'pack_{len(packs)}.pcm\', data))\n            packed.clear()\n        status(\'separate\', 40 + int(54 * min(frames, start + CORE) / frames), parts, frames, packs)\n    write_json(\'result.json\', dict(frames=frames, sample_rate=RATE, channels=2,\n                                  model_sha256=MODEL_SHA, provider=\'cuda\', chunks=parts, packs=packs))\n    status(\'done\', 97, parts, frames, packs)\n\n\ndef run_job(folder):\n    global _PROTOCOL\n    os.chdir(folder)\n    _PROTOCOL = json.loads(Path(\'request.json\').read_text(\'utf-8\')).get(\'protocol\', 1)\n    try:\n        status(\'bootstrap\', 27)\n        run()\n    except Exception as error:\n        report_failure(error)\n        raise\n\n\ndef run_v2(req):\n    # One job for any duration. Only 32 s input / 16 s output packs and 6 s\n    # inference contexts are resident; status and result manifests stay bounded.\n    import numpy as np\n    frames = req.get(\'frames\')\n    stride = 32 * RATE\n    output_stride = 16 * RATE\n    inputs = req.get(\'inputs\')\n    compressed = req.get(\'source\')\n    if compressed is None:\n        if type(frames) is not int or frames <= 0:\n            raise ValueError(\'MUSIC_DURATION\')\n        if not isinstance(inputs, list) or len(inputs) != (frames + stride - 1) // stride:\n            raise ValueError(\'MUSIC_MANIFEST\')\n    pipeline = req.get(\'pipeline\', \'voc_ft\')\n    if pipeline not in (\'voc_ft\', FX_PIPELINE):\n        raise ValueError(\'MUSIC_PIPELINE\')\n    if pipeline == FX_PIPELINE: prepare_fx()\n    else: prepare()\n    if compressed is not None:\n        # The phone sent the clip\'s own compressed audio. When it cannot be read here the job ends without a result\n        # and says so, and the phone sends the decoded audio the long way in its next job; the service stays up.\n        try:\n            frames = decode_source(compressed, req.get(\'seconds\'))\n        except Exception as error:\n            code = str(error) if isinstance(error, ValueError) and re.fullmatch(r\'[A-Z][A-Z0-9_]{0,79}\', str(error)) else type(error).__name__\n            Path(\'input.pcm\').unlink(missing_ok=True)\n            status(\'done\', 97, extra=dict(source_failed=code))\n            return\n    else:\n        with open(\'input.pcm\', \'wb\') as out:\n            for i, part in enumerate(inputs):\n                expected = min(stride, frames - i * stride) * 4\n                if not isinstance(part, dict) or part.get(\'bytes\') != expected:\n                    raise ValueError(\'MUSIC_MANIFEST\')\n                if shutil.disk_usage(\'.\').free < expected + 16 * 1024 * 1024:\n                    raise ValueError(\'MUSIC_DISK\')\n                out.write(read_wire(part, f\'input_{i}.pcm\', expected))\n    if pipeline == FX_PIPELINE:\n        return publish_fx(frames)\n    source = np.memmap(\'input.pcm\', dtype=\'<i2\', mode=\'r\', shape=(frames, 2))\n    recent, packed, count = [], bytearray(), 0\n    began = time.monotonic()\n    gpu_ms = 0\n    for start in range(0, frames, CORE):\n        left, right = max(0, start - RATE), min(frames, start + CORE + RATE)\n        samples = np.ascontiguousarray(source[left:right].T.astype(np.float32) / 32768)\n        infer_at = time.monotonic()\n        output = ENGINE.process(sample_rate=RATE, samples=samples)\n        gpu_ms += (time.monotonic() - infer_at) * 1000\n        if output.sample_rate != RATE or len(output.stems) != 2:\n            raise ValueError(\'MUSIC_OUTPUT_FORMAT\')\n        vocals = output.stems[0].data\n        keep, offset = min(CORE, frames - start), start - left\n        if vocals.ndim != 2 or vocals.shape[0] != 2 or vocals.shape[1] < offset + keep:\n            raise ValueError(\'MUSIC_OUTPUT_LENGTH\')\n        values = vocals[:, offset:offset + keep].T\n        if not np.isfinite(values).all():\n            raise ValueError(\'MUSIC_OUTPUT_FINITE\')\n        packed.extend(np.floor(np.clip(values, -1, 1) * 32767 + .5).astype(\'<i2\').tobytes())\n        if len(packed) == output_stride * 4 or start + CORE >= frames:\n            if shutil.disk_usage(\'.\').free < len(packed) * 3 + 16 * 1024 * 1024:\n                raise ValueError(\'MUSIC_DISK\')\n            meta = write_pack(f\'pack_{count}.pcm\', bytes(packed))\n            meta[\'index\'] = count\n            write_json(f\'pack_{count}.json\', meta)\n            recent.append(meta)\n            recent = recent[-8:]\n            count += 1\n            packed.clear()\n            complete = min(frames, start + CORE)\n            status(\'separate\', 40 + int(54 * complete / frames), extra=dict(\n                frames=frames, completed_frames=complete, sample_rate=RATE, channels=2,\n                model_sha256=MODEL_SHA, pack_count=count, output_frames=output_stride,\n                recent_packs=recent, gpu_ms=round(gpu_ms), inference_ms=round((time.monotonic()-began)*1000)))\n    source._mmap.close()\n    Path(\'input.pcm\').unlink()\n    manifest = dict(protocol=2, job=Path.cwd().name, frames=frames, completed_frames=frames,\n                    sample_rate=RATE, channels=2, model_sha256=MODEL_SHA, provider=\'cuda\',\n                    pack_count=count, output_frames=output_stride, recent_packs=recent,\n                    gpu_ms=round(gpu_ms), inference_ms=round((time.monotonic()-began)*1000))\n    write_json(\'result.json\', manifest)\n    status(\'done\', 97, extra=manifest)\n\n\ndef serve(root):\n    global CACHE, _SERVICE_ROOT, _ACTIVE_JOB, _NEXT_BATCH\n    root = Path(root).resolve()\n    CACHE = _SERVICE_ROOT = root\n    os.chdir(root)\n    config_path = root / \'engine.json\'\n    profile = json.loads(config_path.read_text()).get(\'pipeline\', \'voc_ft\') if config_path.is_file() else \'voc_ft\'\n    if profile == FX_PIPELINE: prepare_fx()\n    elif profile == \'voc_ft\': prepare()\n    else: raise ValueError(\'MUSIC_PIPELINE\')\n    with _STATUS_LOCK:\n        service_state()\n    previous = None\n    while True:\n        job = root.parent / (root.name + \'_b\' + str(_NEXT_BATCH))\n        marker = job / \'submit.json\'\n        legacy = root / \'command.json\'\n        if marker.is_file():\n            try:\n                request = json.loads(marker.read_text(\'utf-8\'))\n            except (OSError, ValueError):\n                # Contents PUT can be in flight; an incomplete marker isn\'t executable.\n                time.sleep(.2)\n                continue\n            if request != dict(protocol=2, service=root.name, job=job.name):\n                raise ValueError(\'MUSIC_JOB_PATH\')\n            try:\n                terminal = json.loads((job / \'progress.json\').read_text(\'utf-8\'))\n            except (OSError, ValueError):\n                terminal = {}\n            if terminal.get(\'stage\') in (\'done\', \'failed\'):\n                _NEXT_BATCH += 1\n                continue\n            write_json(job / \'accepted.json\', dict(job=job.name, protocol=2))\n            _NEXT_BATCH += 1\n        elif legacy.is_file():\n            request = json.loads(legacy.read_text(\'utf-8\'))\n            job = Path(request[\'folder\']).resolve()\n            legacy.unlink()\n        else:\n            time.sleep(.2)\n            continue\n        if job.parent != root.parent or not re.fullmatch(re.escape(root.name) + r\'_b[0-9]+\', job.name):\n            raise ValueError(\'MUSIC_JOB_PATH\')\n        if previous is not None and previous != job:\n            shutil.rmtree(previous, ignore_errors=True)\n        _ACTIVE_JOB = job.name\n        with _STATUS_LOCK:\n            service_state()\n        run_job(job)\n        _ACTIVE_JOB = \'\'\n        with _STATUS_LOCK:\n            service_state()\n        previous = job\n\n\nif __name__ == \'__main__\':\n    stop = threading.Event()\n    thread = threading.Thread(target=heartbeat, args=(stop,), daemon=True)\n    thread.start()\n    try:\n        if len(sys.argv) == 3 and sys.argv[2] == \'--serve\':\n            serve(sys.argv[1])\n        else:\n            run_job(Path(sys.argv[1]).resolve())\n    except Exception as error:\n        report_failure(error)\n        raise\n    finally:\n        stop.set()\n        thread.join(timeout=3)\n', 'music_launch.py': '"""Prepare private cuDNN, then exec a clean worker before it imports torch."""\nimport importlib.util\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport uuid\n\nPIN = \'9.13.0.50\'\n\ndef cuda_major():\n    # Reading wheel metadata avoids importing torch/CUDA twice in fresh processes.\n    from importlib.metadata import version\n    wheel = version(\'torch\')\n    match = re.search(r\'\\+cu(\\d{2,3})\', wheel)\n    if match:\n        return int(match.group(1)[:2])\n    if \'+cpu\' in wheel:\n        return 0\n    spec = importlib.util.find_spec(\'torch\')\n    if spec is not None and spec.origin:\n        text = Path(spec.origin).with_name(\'version.py\').read_text()\n        match = re.search(r"(?m)^cuda(?:\\s*:[^=\\n]+)?\\s*=\\s*[\'\\"](\\d+)", text)\n        if match:\n            return int(match.group(1))\n        if re.search(r\'(?m)^cuda(?:\\s*:[^=\\n]+)?\\s*=\\s*None\', text):\n            return 0\n    raise ValueError(\'MUSIC_GPU_WHEEL\')\n\ndef launch(root):\n    root = Path(root).resolve()\n    if not re.fullmatch(r\'madar_[a-f0-9]{32}\', root.name):\n        raise ValueError(\'MUSIC_JOB_PATH\')\n    os.chdir(root)\n    spec = importlib.util.spec_from_file_location(\'madar_music_status\', root/\'music_worker.py\')\n    worker = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(worker)\n    stop = threading.Event()\n    thread = threading.Thread(target=worker.heartbeat, args=(stop,), daemon=True)\n    thread.start()\n    try:\n        worker.status(\'gpu_check\', 27)\n        major = cuda_major()\n        env = os.environ.copy()\n        env[\'CUDNN_LIB_CONFIG\'] = \'FULL\'\n        env.pop(\'MADAR_CUDNN_LIB\', None)\n        if major == 13:\n            worker.status(\'gpu_dependencies\', 28)\n            cache = root.parent/\'madar_cudnn_9130050\'\n            libs = cache/\'nvidia/cudnn/lib\'\n            required = [\'libcudnn.so.9\', \'libcudnn_engines_precompiled.so.9\']\n            if not all((libs/name).is_file() for name in required):\n                staging = root.parent/(\'madar_cudnn_stage_\'+uuid.uuid4().hex)\n                try:\n                    subprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'--disable-pip-version-check\', \'--no-deps\', \'--only-binary=:all:\', \'--timeout\', \'20\', \'--retries\', \'2\', \'--target\', str(staging), \'nvidia-cudnn-cu13==\'+PIN], check=True, timeout=600)\n                    if not all((staging/\'nvidia/cudnn/lib\'/name).is_file() for name in required):\n                        raise ValueError(\'MUSIC_CUDNN_INSTALL\')\n                    if cache.exists():shutil.rmtree(cache)\n                    staging.replace(cache)\n                finally:\n                    if staging.exists():shutil.rmtree(staging)\n            env[\'MADAR_CUDNN_LIB\'] = str(libs/\'libcudnn.so.9\')\n            env[\'LD_LIBRARY_PATH\'] = str(libs)+os.pathsep+env.get(\'LD_LIBRARY_PATH\', \'\')\n        # No torch import in the launcher. The clean worker imports only\n        # after the pinned native library has been preloaded by music_worker.py.\n        stop.set();thread.join(timeout=3)\n        os.execve(sys.executable, [sys.executable, str(root/\'music_worker.py\'), str(root), \'--serve\'], env)\n    except Exception as error:\n        worker.report_failure(error)\n        raise\n    finally:\n        stop.set();thread.join(timeout=3)\n\nif __name__ == \'__main__\':\n    launch(sys.argv[1])\n', 'music_warmup.py': '"""Start owned GPU/model setup while Android decodes and uploads the first window."""\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\nservice = SERVICE_FOLDER\nif not isinstance(service, str) or not re.fullmatch(r\'madar_[a-f0-9]{32}\', service):\n    raise ValueError(\'MADAR_JOB_ID\')\ncurrent = Path.cwd()\nroots = ([current] if current.name == service else []) + [current / service, current.parent / service, Path(\'/content\') / service, Path(\'/\') / service]\nroot = next((p.resolve() for p in roots if (p / \'music_worker.py\').is_file()), None)\nif root is None:\n    raise FileNotFoundError(\'MADAR_JOB_FILES\')\nif \'_madar_music_service\' not in globals() or _madar_music_service.poll() is not None:\n    _madar_music_log = (root / \'worker.log\').open(\'ab\')\n    launcher = root / \'music_launch.py\'\n    command = [sys.executable, str(launcher), str(root)] if launcher.is_file() else [sys.executable, str(root / \'music_worker.py\'), str(root), \'--serve\']\n    _madar_music_service = subprocess.Popen(command, stdout=_madar_music_log, stderr=_madar_music_log)\n', 'music_bootstrap.py': "import os\nfrom pathlib import Path\nimport subprocess\nimport sys\n\nfolder = JOB_FOLDER\nimport re\nif not isinstance(folder, str) or not re.fullmatch(r'madar_[A-Za-z0-9]+(?:_b[0-9]+)?', folder):\n    raise ValueError('MADAR_JOB_ID')\ncurrent = Path.cwd()\npaths = ([current] if current.name == folder else []) + [current / folder, current.parent / folder, Path('/content') / folder, Path('/') / folder]\njob = next((p.resolve() for p in paths if (p / 'request.json').is_file()), None)\nif job is None:\n    raise FileNotFoundError('MADAR_JOB_FILES')\nif 'SERVICE_FOLDER' in globals():\n    import json\n    import re\n    import time\n    service = SERVICE_FOLDER\n    if not isinstance(service, str) or not re.fullmatch(r'madar_[a-f0-9]{32}', service):\n        raise ValueError('MADAR_JOB_ID')\n    roots = ([current] if current.name == service else []) + [current / service, current.parent / service, Path('/content') / service, Path('/') / service]\n    root = next((p.resolve() for p in roots if (p / 'music_worker.py').is_file()), None)\n    if root is None or job.parent != root.parent or not re.fullmatch(re.escape(root.name) + r'_b[0-9]+', job.name):\n        raise ValueError('MUSIC_JOB_PATH')\n    if '_madar_music_service' not in globals() or _madar_music_service.poll() is not None:\n        try:\n            failed = json.loads((root / 'progress.json').read_text())\n        except (OSError, ValueError):\n            failed = {}\n        code = failed.get('error', 'MUSIC_PROCESS_EXIT')\n        if not isinstance(code, str) or not re.fullmatch(r'[A-Za-z][A-Za-z0-9_]{0,79}', code):\n            code = 'MUSIC_PROCESS_EXIT'\n        (job / 'progress.json').write_text(json.dumps(dict(stage='failed', progress=0, error=code)))\n        raise RuntimeError(code)\n    command = root / 'command.tmp'\n    command.write_text(json.dumps(dict(folder=str(job))))\n    command.replace(root / 'command.json')\n    deadline = time.monotonic() + 1800\n    while True:\n        try:\n            state = json.loads((job / 'progress.json').read_text())\n        except (OSError, ValueError):\n            state = {}\n        if state.get('stage') in ('done', 'failed'):\n            break\n        exit_code = _madar_music_service.poll()\n        if exit_code is not None or time.monotonic() > deadline:\n            # The child may publish its precise failure between the read above and poll().\n            # Re-read after observing exit; never replace a verified terminal state.\n            try:\n                terminal = json.loads((job / 'progress.json').read_text())\n            except (OSError, ValueError):\n                terminal = {}\n            if terminal.get('stage') in ('done', 'failed'):\n                break\n            code = 'MUSIC_TIMEOUT' if exit_code is None else 'NATIVE_ABORT' if exit_code < 0 else 'PROCESS_EXIT'\n            if exit_code is None:\n                _madar_music_service.kill()\n                _madar_music_service.wait(timeout=5)\n            value = dict(stage='failed', last_stage=terminal.get('stage', 'bootstrap'), progress=0, error=code)\n            if type(exit_code) is int:\n                value['exit_code'] = exit_code\n            (job / 'progress.json').write_text(json.dumps(value))\n            raise RuntimeError(code)\n        time.sleep(.2)\nelse:\n    # Standalone export/diagnostic compatibility path.\n    with (job / 'worker.log').open('wb') as log:\n        subprocess.run([sys.executable, str(job / 'music_worker.py'), str(job)], stdout=log, stderr=log, check=True, timeout=1800)\n", 'music_profiles.py': '"""Versioned, explicit profiles. An omitted profile always means the existing model."""\nimport hashlib\nimport json\n\nLEGACY = \'voc_ft\'\nFX = \'inst_hq_4_3\'\nLEGACY_SHA = \'e411182ce2c53541fefdcc99a8f46f2fe03978eb22038b9497c1d9d95a00fad4\'\nMODELS = (\n    (\'UVR-MDX-NET-Inst_HQ_4.onnx\', \'3c4b5b9b05090fdf238f38ba5046813982d50e2a652e9cb3324ea79720c3c9c8\'),\n    (\'UVR-MDX-NET-Inst_HQ_3.onnx\', \'317554b07fe1ea5279a77f2b1520a41ea4b93432560c4ffd08792c30fddf9adc\'),\n)\nCONFIG = dict(pipeline=FX, models=MODELS, batch_size=8, overlap=0.1,\n              segment_size=256, sample_rate=44100, output_stem=\'Vocals\', audio_separator=\'0.48.0\', librosa=\'0.11.0\')\nFX_SHA = hashlib.sha256(json.dumps(CONFIG, sort_keys=True, separators=(\',\', \':\')).encode()).hexdigest()\n\ndef validate(value):\n    if value not in (LEGACY, FX):\n        raise ValueError(\'MUSIC_PIPELINE\')\n    return value\n\ndef fingerprint(value):\n    return LEGACY_SHA if validate(value) == LEGACY else FX_SHA\n', 'music_fx_bridge.py': '"""Isolated optional dependencies and one retained two-model child per GPU session."""\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport site\nimport subprocess\nimport sys\nimport time\nimport venv\n\nfrom music_profiles import CONFIG, FX_SHA\n\ndef create_environment(folder):\n    # Colab\'s Debian Python may not include ensurepip/python3-venv. Its\n    # already-running pip can manage a bare venv using --python (pip >=22.3).\n    # Never bootstrap pip from the network or modify the notebook\'s packages.\n    try:\n        venv.EnvBuilder(with_pip=False, system_site_packages=True).create(folder)\n        # A nested venv normally sees the base interpreter, not its parent\'s\n        # venv packages. Preserve Colab\'s exact installed Torch/CUDA packages\n        # even when the notebook interpreter itself lives in a venv. The child\n        # site-packages stays first, so optional installs remain isolated.\n        library = Path(folder)/\'lib\'/(\'python\'+str(sys.version_info.major)+\'.\'+str(sys.version_info.minor))/\'site-packages\'\n        parents = [str(Path(path).resolve()) for path in site.getsitepackages() if Path(path).is_dir()]\n        (library/\'madar-parent-site.pth\').write_text(\'\\n\'.join(parents)+\'\\n\')\n    except (Exception, SystemExit) as error:\n        raise ValueError(\'MUSIC_FX_ENV\') from error\n\ndef pip_command(folder):\n    return [sys.executable, \'-m\', \'pip\', \'--python\', str(Path(folder)/\'bin/python\')]\n\ndef install(command, log, **options):\n    try:\n        with Path(log).open(\'wb\') as output:\n            subprocess.run(command, stdout=output, stderr=output, check=True, **options)\n    except subprocess.TimeoutExpired as error:\n        raise ValueError(\'MUSIC_FX_INSTALL_TIMEOUT\') from error\n    except subprocess.CalledProcessError as error:\n        # Only fixed codes leave Colab; pip logs can contain private index URLs.\n        raise ValueError(\'MUSIC_FX_INSTALL\') from error\n\nclass Client:\n    def __init__(self, root, status):\n        self.root, self.status = Path(root).resolve(), status\n        self.folder = self.root / \'fx_rpc\'\n        self.folder.mkdir(exist_ok=True)\n        self.sequence, self.process, self.last = 0, None, None\n\n    def prepare(self):\n        if self.process is not None:\n            if self.process.poll() is not None:\n                raise ValueError(\'MUSIC_FX_INFERENCE\')\n            return\n        self.status(\'install\', 28)\n        from music_launch import cuda_major\n        major = cuda_major()\n        if major not in (12, 13):\n            raise ValueError(\'MUSIC_GPU_WHEEL\')\n        torch_version = importlib.metadata.version(\'torch\')\n        ort = \'1.28.0\' if major == 13 else \'1.22.0\'\n        identity = dict(audio_separator=CONFIG[\'audio_separator\'], ort=ort, torch=torch_version,\n                        python=list(sys.version_info[:2]), numpy=\'2.2.6\', librosa=CONFIG[\'librosa\'], bootstrap=2)\n        envroot = self.root.parent / (\'madar_fx_env_cu\' + str(major) + \'_0480_\' + str(sys.version_info.minor))\n        marker = envroot / \'madar-ready.json\'\n        try: installed = json.loads(marker.read_text()) == identity\n        except (OSError, ValueError): installed = False\n        if not installed:\n            if envroot.exists(): shutil.rmtree(envroot)\n            create_environment(envroot)\n            constraints = envroot / \'constraints.txt\'\n            constraints.write_text(\'torch==\'+torch_version+\'\\nnumpy==2.2.6\\nonnxruntime-gpu==\'+ort+\'\\n\')\n            install_env = os.environ.copy()\n            if shutil.which(\'gcc\'): install_env[\'CC\'] = \'gcc\'\n            if shutil.which(\'g++\'): install_env[\'CXX\'] = \'g++\'\n            install(pip_command(envroot) + [\'install\', \'--disable-pip-version-check\',\n                    \'--timeout\', \'30\', \'--retries\', \'2\', \'-c\', str(constraints),\n                    \'audio-separator[gpu]==\'+CONFIG[\'audio_separator\'], \'onnxruntime-gpu==\'+ort, \'numpy==2.2.6\', \'librosa==\'+CONFIG[\'librosa\'], \'audioread==3.0.1\'],\n                    self.root / \'fx-install.log\', timeout=600, env=install_env)\n            marker.write_text(json.dumps(identity))\n        if shutil.which(\'ffmpeg\') is None:\n            with (self.root / \'fx-ffmpeg.log\').open(\'wb\') as log:\n                subprocess.run([\'apt-get\', \'update\', \'-qq\'], stdout=log, stderr=log, check=True, timeout=180)\n                subprocess.run([\'apt-get\', \'install\', \'-y\', \'-qq\', \'ffmpeg\'], stdout=log, stderr=log, check=True, timeout=180)\n        # The launcher already pins CUDA 13 cuDNN. Pin the CUDA 12 variant too,\n        # privately, without changing torch or the old model\'s site-packages.\n        env = os.environ.copy()\n        env[\'CUDNN_LIB_CONFIG\'] = \'FULL\'\n        env.pop(\'PYTHONPATH\', None)\n        if major == 12:\n            cudnn = self.root.parent/\'madar_fx_cudnn_cu12_9130050\'\n            libs = cudnn/\'nvidia/cudnn/lib\'\n            if not (libs/\'libcudnn.so.9\').is_file():\n                install(pip_command(envroot) + [\'install\', \'--no-deps\',\n                        \'--only-binary=:all:\', \'--target\', str(cudnn), \'nvidia-cudnn-cu12==9.13.0.50\'],\n                        self.root/\'fx-cudnn.log\', timeout=600)\n            env[\'MADAR_CUDNN_LIB\'] = str(libs/\'libcudnn.so.9\')\n            env[\'LD_LIBRARY_PATH\'] = str(libs)+os.pathsep+env.get(\'LD_LIBRARY_PATH\', \'\')\n        # An old status cannot satisfy a newly started child.\n        (self.folder/\'state.json\').unlink(missing_ok=True)\n        self.log = (self.root/\'fx-worker.log\').open(\'ab\')\n        self.process = subprocess.Popen([str(envroot/\'bin/python\'), str(self.root/\'music_fx_engine.py\'),\n            str(self.root), str(os.getpid())], stdout=self.log, stderr=self.log, env=env)\n        self.wait(lambda state: state.get(\'stage\') == \'ready\', 600)\n\n    def wait(self, completed, timeout):\n        began, heartbeat_at, previous_beat = time.monotonic(), time.monotonic(), None\n        while True:\n            if self.process.poll() is not None:\n                raise ValueError(\'MUSIC_FX_INFERENCE\')\n            try: state = json.loads((self.folder/\'state.json\').read_text())\n            except (OSError, ValueError): state = {}\n            beat = state.get(\'heartbeat\')\n            if beat != previous_beat:\n                previous_beat, heartbeat_at = beat, time.monotonic()\n            if state.get(\'stage\') == \'failed\':\n                raise ValueError(state.get(\'error\', \'MUSIC_FX_INFERENCE\'))\n            if completed(state): return state\n            stage, detail = state.get(\'stage\'), state.get(\'detail\', \'\')\n            if stage in (\'model_download\', \'model_load\', \'gpu_inference_check\', \'separate\') and (stage, detail) != self.last:\n                self.last = (stage, detail)\n                # Heartbeat and status remain bounded; no stage-1 PCM is published.\n                self.status(stage, 40 if detail == \'pass1\' else 70 if detail == \'pass2\' else 32)\n            if time.monotonic()-began > timeout or time.monotonic()-heartbeat_at > 120:\n                self.process.terminate()\n                self.process.wait(timeout=10)\n                raise ValueError(\'MUSIC_TIMEOUT\')\n            time.sleep(.2)\n\n    def run(self, job, frames):\n        self.prepare()\n        self.sequence += 1\n        command = dict(sequence=self.sequence, job=str(Path(job).resolve()), frames=frames)\n        path = self.folder/\'command.tmp\'\n        path.write_text(json.dumps(command));path.replace(self.folder/\'command.json\')\n        result = self.wait(lambda state: state.get(\'stage\') == \'done\' and state.get(\'sequence\') == self.sequence, 1800)\n        if result.get(\'pipeline\') != CONFIG[\'pipeline\'] or result.get(\'model_sha256\') != FX_SHA:\n            raise ValueError(\'MUSIC_RESULT_SCHEMA\')\n        output = Path(job)/\'fx_final.pcm\'\n        if not output.is_file() or output.stat().st_size != frames*4:\n            raise ValueError(\'MUSIC_OUTPUT_LENGTH\')\n        return output, result[\'gpu_ms\']\n', 'music_fx_engine.py': '"""HQ4 -> HQ3, cached CUDA models. Only the final residual is returned.\n\nThis process lives in an isolated Python environment; it never installs into\nthe existing Voc_FT runtime. The CPU switch is for explicit offline QA only.\n"""\nimport hashlib\nimport json\nimport logging\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\nimport tempfile\nimport threading\nimport time\nimport wave\n\nfrom music_profiles import CONFIG, FX, FX_SHA, MODELS\n\nRATE = 44100\n\ndef atomic(path, value):\n    path = Path(path)\n    tmp = path.with_suffix(\'.tmp\')\n    tmp.write_text(json.dumps(value), encoding=\'utf-8\')\n    tmp.replace(path)\n\ndef check_output(paths, directory):\n    # Use the API\'s current return value, never a glob that can select an old file.\n    if not isinstance(paths, list) or len(paths) != 1:\n        raise ValueError(\'MUSIC_FX_STEM\')\n    path = Path(paths[0])\n    if not path.is_absolute():\n        path = directory / path\n    path = path.resolve()\n    if path.parent != directory.resolve() or not path.is_file() or path.suffix.lower() != \'.wav\':\n        raise ValueError(\'MUSIC_FX_STEM\')\n    return path\n\nclass Engine:\n    def __init__(self, cache, status=lambda *args: None, require_gpu=True):\n        # Must happen before torch/ORT import, in a fresh child process.\n        if os.environ.get(\'MADAR_CUDNN_LIB\'):\n            import ctypes\n            self.cudnn = ctypes.CDLL(os.environ[\'MADAR_CUDNN_LIB\'], mode=ctypes.RTLD_GLOBAL)\n            self.cudnn.cudnnGetVersion.restype = ctypes.c_size_t\n            if self.cudnn.cudnnGetVersion() != 91300:\n                raise ValueError(\'MUSIC_CUDNN_VERSION\')\n        import numpy as np\n        import torch\n        import onnxruntime as ort\n        if require_gpu and (not torch.cuda.is_available() or \'CUDAExecutionProvider\' not in ort.get_available_providers()):\n            raise ValueError(\'MUSIC_GPU_UNAVAILABLE\')\n        self.np, self.torch, self.require_gpu = np, torch, require_gpu\n        self.cache = Path(cache).resolve()\n        self.cache.mkdir(parents=True, exist_ok=True)\n        self.separators, self.sessions = [], []\n        original = ort.InferenceSession\n        def session(*args, **kwargs):\n            options = kwargs.get(\'sess_options\') or ort.SessionOptions()\n            options.intra_op_num_threads = 2\n            options.inter_op_num_threads = 1\n            options.enable_profiling = True\n            options.profile_file_prefix = str(self.cache / (\'probe_\' + str(len(self.sessions))))\n            kwargs[\'sess_options\'] = options\n            value = original(*args, **kwargs)\n            if require_gpu and \'CUDAExecutionProvider\' not in value.get_providers():\n                raise ValueError(\'MUSIC_GPU_PROVIDER\')\n            self.sessions.append(value)\n            return value\n        ort.InferenceSession = session\n        try:\n            from audio_separator.separator import Separator\n            for name, expected in MODELS:\n                status(\'model_download\', name)\n                separator = Separator(model_file_dir=str(self.cache), output_dir=str(self.cache),\n                    output_format=\'WAV\', output_single_stem=\'Vocals\', use_soundfile=True,\n                    log_level=logging.WARNING, sample_rate=RATE,\n                    mdx_params=dict(hop_length=1024, segment_size=256, overlap=0.1,\n                                    batch_size=8, enable_denoise=False))\n                separator.load_model(model_filename=name)\n                model = self.cache / name\n                if not model.is_file() or hashlib.file_digest(model.open(\'rb\'), \'sha256\').hexdigest() != expected:\n                    raise ValueError(\'MODEL_HASH\')\n                instance = separator.model_instance\n                if (instance.primary_stem_name != \'Instrumental\' or instance.secondary_stem_name != \'Vocals\'\n                        or getattr(instance, \'uses_pytorch_inference\', True)):\n                    raise ValueError(\'MUSIC_FX_STEM\')\n                self.separators.append(separator)\n        finally:\n            ort.InferenceSession = original\n        if len(self.sessions) != 2:\n            raise ValueError(\'MUSIC_FX_STEM\')\n        status(\'gpu_inference_check\', \'\')\n        # Profile actual operations in BOTH models. Provider presence alone is insufficient.\n        with tempfile.TemporaryDirectory(prefix=\'fx_probe_\', dir=self.cache) as temporary:\n            folder = Path(temporary)\n            source = folder / \'input.wav\'\n            import soundfile as sf\n            t = np.arange(2 * RATE, dtype=np.float32) / RATE\n            samples = np.stack((0.01*np.sin(2*np.pi*440*t), 0.01*np.sin(2*np.pi*660*t)), axis=1)\n            sf.write(source, samples, RATE, subtype=\'PCM_16\')\n            self.separate(source, folder, lambda *args: None)\n        for value in self.sessions:\n            profile = Path(value.end_profiling())\n            events = json.loads(profile.read_text())\n            profile.unlink(missing_ok=True)\n            if require_gpu and not any(e.get(\'args\', {}).get(\'provider\') == \'CUDAExecutionProvider\' and any(op in e.get(\'args\', {}).get(\'op_name\', \'\').lower() for op in (\'conv\', \'gemm\', \'matmul\')) for e in events):\n                raise ValueError(\'MUSIC_GPU_PROVIDER\')\n        status(\'ready\', \'\')\n\n    def separate(self, source, folder, status):\n        result = Path(source)\n        for index, separator in enumerate(self.separators):\n            directory = Path(folder) / (\'pass\' + str(index + 1))\n            directory.mkdir(exist_ok=True)\n            separator.output_dir = str(directory)\n            separator.model_instance.output_dir = str(directory)\n            status(\'separate\', \'pass\' + str(index + 1))\n            paths = separator.separate(str(result), custom_output_names={\'Vocals\': \'dialogue_fx\'})\n            result = check_output(paths, directory)\n        return result\n\n    def process_pcm(self, job, frames, status):\n        import soundfile as sf\n        source, output = Path(job) / \'input.pcm\', Path(job) / \'fx_final.pcm\'\n        if type(frames) is not int or frames <= 0 or source.stat().st_size != frames * 4:\n            raise ValueError(\'MUSIC_OUTPUT_LENGTH\')\n        with tempfile.TemporaryDirectory(prefix=\'two_pass_\', dir=job) as temporary:\n            folder = Path(temporary)\n            wav = folder / \'input.wav\'\n            # PCM is already decoded by Android; don\'t upload or re-decode the video.\n            with source.open(\'rb\') as stream, wave.open(str(wav), \'wb\') as out:\n                out.setnchannels(2); out.setsampwidth(2); out.setframerate(RATE)\n                while True:\n                    block = stream.read(1024 * 1024)\n                    if not block: break\n                    out.writeframesraw(block)\n            infer_at = time.monotonic()\n            result = self.separate(wav, folder, status)\n            self.last_inference_ms = round((time.monotonic()-infer_at)*1000)\n            info = sf.info(result)\n            if info.samplerate != RATE or info.channels != 2 or info.frames != frames:\n                raise ValueError(\'MUSIC_OUTPUT_LENGTH\')\n            temporary_pcm = output.with_suffix(\'.partial\')\n            try:\n                with sf.SoundFile(result) as stream, temporary_pcm.open(\'wb\') as out:\n                    while True:\n                        values = stream.read(65536, dtype=\'float32\', always_2d=True)\n                        if len(values) == 0: break\n                        if not self.np.isfinite(values).all():\n                            raise ValueError(\'MUSIC_OUTPUT_FINITE\')\n                        out.write(self.np.floor(self.np.clip(values, -1, 1)*32767 + .5).astype(\'<i2\').tobytes())\n                if temporary_pcm.stat().st_size != frames * 4:\n                    raise ValueError(\'MUSIC_OUTPUT_LENGTH\')\n                temporary_pcm.replace(output)\n            finally:\n                temporary_pcm.unlink(missing_ok=True)\n        return output\n\ndef serve(root, owner_pid):\n    root = Path(root).resolve()\n    if not re.fullmatch(r\'madar_[a-f0-9]{32}\', root.name):\n        raise ValueError(\'MUSIC_JOB_PATH\')\n    folder = root / \'fx_rpc\'; folder.mkdir(exist_ok=True)\n    state, lock, finished = {}, threading.RLock(), threading.Event()\n    beat = 0\n    def publish(stage, detail=\'\', **extra):\n        with lock:\n            state.clear(); state.update(stage=stage, detail=detail, **extra)\n            atomic(folder / \'state.json\', dict(state, heartbeat=beat))\n    def heartbeat():\n        nonlocal beat\n        while not finished.wait(.2):\n            try: request = json.loads((root / \'stop.json\').read_text())\n            except (OSError, ValueError): request = {}\n            if os.getppid() != owner_pid or request == dict(service=root.name, cancel=True):\n                os._exit(0)\n            with lock:\n                beat += 1\n                atomic(folder / \'state.json\', dict(state, heartbeat=beat))\n    thread = threading.Thread(target=heartbeat, daemon=True); thread.start()\n    try:\n        engine = Engine(root / \'fx_models\', publish)\n        publish(\'ready\')\n        previous = -1\n        while True:\n            try: command = json.loads((folder / \'command.json\').read_text())\n            except (OSError, ValueError): command = {}\n            sequence = command.get(\'sequence\')\n            if type(sequence) is not int or sequence <= previous:\n                time.sleep(.1); continue\n            job = Path(command.get(\'job\', \'\')).resolve()\n            if job.parent != root.parent or not re.fullmatch(re.escape(root.name) + r\'_b[0-9]+\', job.name):\n                raise ValueError(\'MUSIC_JOB_PATH\')\n            previous = sequence\n            began = time.monotonic()\n            engine.process_pcm(job, command.get(\'frames\'), publish)\n            publish(\'done\', sequence=sequence, gpu_ms=engine.last_inference_ms,\n                    pipeline=FX, model_sha256=FX_SHA)\n    except Exception as error:\n        code = str(error) if isinstance(error, ValueError) and re.fullmatch(r\'[A-Z][A-Z0-9_]{0,79}\', str(error)) else \'MUSIC_FX_INFERENCE\'\n        publish(\'failed\', error=code)\n        raise\n    finally:\n        finished.set(); thread.join(timeout=1)\n\nif __name__ == \'__main__\':\n    serve(sys.argv[1], int(sys.argv[2]))\n'}
for name, source in SOURCES.items():
    (ROOT/name).write_text(source, encoding="utf-8")
(ROOT/"engine.json").write_text(json.dumps({"pipeline": PIPELINE}))
sys.path.insert(0, str(ROOT))
from music_profiles import validate, fingerprint
validate(PIPELINE)
print("كتب الكود الكامل:", ", ".join(SOURCES))


In [ ]:
#@title 3. رفع فيديو أو ملف صوتي
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("اختر ملفًا واحدًا للتجربة")
INPUT = (Path.cwd()/Path(next(iter(uploaded))).name).resolve()
del uploaded
if not INPUT.is_file(): raise FileNotFoundError(INPUT.name)
if not __import__('shutil').which('ffmpeg'):
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(['apt-get','install','-y','-qq','ffmpeg'], check=True)
probe=json.loads(subprocess.check_output(['ffprobe','-v','error','-show_streams','-of','json',str(INPUT)]))
HAS_VIDEO=any(s.get('codec_type')=='video' for s in probe['streams'])
# Trim once, accurately; both models and all previews receive this same part.
SEGMENT=RESULTS/('original_segment.mkv' if HAS_VIDEO else 'original_segment.wav')
command=['ffmpeg','-v','error','-y','-i',str(INPUT),'-ss',str(START_SECONDS),'-t',str(DURATION_SECONDS)]
if HAS_VIDEO:
    command+=['-map','0:v:0','-map','0:a:0','-c:v','libx264','-preset','fast','-crf','18','-pix_fmt','yuv420p']
else: command+=['-vn','-map','0:a:0']
command+=['-c:a','pcm_s16le','-ar','48000','-ac','2',str(SEGMENT)]
subprocess.run(command,check=True)
FAST_WAV = JOB/'fast_input.wav' 
subprocess.run(['ffmpeg','-y','-i',str(SEGMENT),'-vn','-acodec','pcm_s16le',
                '-ar','44100','-ac','2',str(FAST_WAV),'-loglevel','error'], check=True)
inputs=[]
with wave.open(str(FAST_WAV),'rb') as audio:
    FRAMES=audio.getnframes()
    if FRAMES<=0: raise ValueError("المقطع لا يحتوي على صوت")
    while True:
        raw=audio.readframes(32*44100)
        if not raw: break
        compressed=zlib.compress(raw,1)
        meta={'bytes':len(raw),'sha256':hashlib.sha256(raw).hexdigest()}
        name=JOB/('input_'+str(len(inputs))+'.pcm')
        if len(compressed)+512<len(raw):
            meta.update(wire_encoding='zlib',wire_bytes=len(compressed),wire_sha256=hashlib.sha256(compressed).hexdigest())
            Path(str(name)+'.zlib').write_bytes(compressed)
        else: name.write_bytes(raw)
        inputs.append(meta)
(JOB/'request.json').write_text(json.dumps(dict(protocol=2,pipeline=PIPELINE,frames=FRAMES,inputs=inputs)))
(JOB/'submit.json').write_text(json.dumps(dict(protocol=2,service=SERVICE_ID,job=JOB.name)))
print(f"الصوت: {FRAMES/44100:.2f} ثانية. الأصل محفوظ: {INPUT.name}")


In [ ]:
#@title 4. تشغيل العامل نفسه والمرحلتين ثم التحقق من النتيجة
worker_log=(ROOT/'notebook-worker.log').open('wb')
process=subprocess.Popen([sys.executable,str(ROOT/'music_launch.py'),str(ROOT)],stdout=worker_log,stderr=worker_log)
started=time.monotonic(); previous=None
try:
    while True:
        state={}
        for path in (ROOT/'progress.json',JOB/'progress.json'):
            try: candidate=json.loads(path.read_text())
            except (OSError,ValueError): continue
            if candidate: state=candidate
        stage=state.get('stage','bootstrap')
        marker=(stage,state.get('progress'))
        if marker!=previous:
            print(round(time.monotonic()-started,1),'s',stage,state.get('progress',''),flush=True)
            previous=marker
        if stage=='failed': raise RuntimeError(state.get('error','فشل الفصل'))
        if stage=='done': break
        if process.poll() is not None: raise RuntimeError((ROOT/'notebook-worker.log').read_text(errors='replace')[-6000:])
        if time.monotonic()-started>3600: raise TimeoutError('انتهت مهلة التجربة')
        time.sleep(.5)
    result=json.loads((JOB/'result.json').read_text())
    assert result['model_sha256']==fingerprint(PIPELINE)
    assert result.get('pipeline','voc_ft')==PIPELINE
    assert result['provider']=='cuda' and result['completed_frames']==FRAMES
    FINAL_WAV=RESULTS/'current_hq4_hq3.wav'
    with wave.open(str(FINAL_WAV),'wb') as audio:
        audio.setnchannels(2);audio.setsampwidth(2);audio.setframerate(44100)
        total=0
        for i in range(result['pack_count']):
            raw=(JOB/f'pack_{i}.pcm').read_bytes()
            meta=json.loads((JOB/f'pack_{i}.json').read_text())
            assert len(raw)==meta['bytes'] and hashlib.sha256(raw).hexdigest()==meta['sha256']
            total+=len(raw);audio.writeframesraw(raw)
        assert total==FRAMES*4
    print('اكتمل الفصل والتحقق. الزمن مع التهيئة:',round(time.monotonic()-started,2),'ثانية')
    print('زمن الفصل:',result.get('gpu_ms',0)/1000,'ثانية')
finally:
    (ROOT/'stop.json').write_text(json.dumps(dict(service=SERVICE_ID,cancel=True)))
    try: process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        process.terminate()
        try: process.wait(timeout=5)
        except subprocess.TimeoutExpired: process.kill();process.wait()
    worker_log.close()

BASELINE_RESULT=result
BASELINE_SECONDS=result.get("gpu_ms",0)/1000


In [ ]:
#@title 5. تجهيز المرشح في بيئة مستقلة وإعادة استخدام Torch الخاص بـ Colab
import venv
BANDIT_REV='96ea0ec12aeb444f379c2f5145fb8ee58b58cf19'
BANDIT_ENV=ROOT/'bandit_env'
venv.EnvBuilder(with_pip=False,system_site_packages=True).create(BANDIT_ENV)
BPY=str(BANDIT_ENV/'bin/python')
# Do not upgrade/downgrade Colab's Torch/CUDA or share the baseline's dependencies.
# Host pip targets the child interpreter; no ensurepip or global package changes.
subprocess.run([sys.executable,'-m','pip','--python',BPY,'install','--disable-pip-version-check','--no-deps',
    'bandit-infer @ git+https://github.com/openmirlab/bandit-infer.git@'+BANDIT_REV,
    'soundfile==0.13.1'],check=True)
subprocess.run([BPY,'-c',"import torch, soundfile, numpy; assert torch.cuda.is_available(), 'اختر GPU للجلسة'; print('Torch:',torch.__version__,'GPU:',torch.cuda.get_device_name(0))"],check=True)


In [ ]:
#@title 6. الفصل مع تقدم التنزيل والأجزاء
RUNNER=ROOT/"bandit_runner.py"
RUNNER.write_text('"""Pinned Bandit runner with verified, resumable downloads and phase/chunk progress."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport socket\nimport sys\nimport time\nimport urllib.request\n\nSTATE = None\nLAST_UPDATE = 0\n\ndef progress(stage, completed=0, total=0, force=False):\n    global LAST_UPDATE\n    now=time.monotonic()\n    if not force and now-LAST_UPDATE<.5: return\n    LAST_UPDATE=now\n    value=dict(stage=stage,completed=completed,total=total,updated=now)\n    if STATE:\n        temporary=STATE.with_suffix(\'.tmp\')\n        temporary.write_text(json.dumps(value,ensure_ascii=False));temporary.replace(STATE)\n    print(stage,completed,\'/\',total,flush=True)\n\n\ndef ensure_weights(folder, spec):\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\n    target=folder/spec.filename\n    def valid(path):\n        if not path.is_file() or path.stat().st_size!=spec.size:return False\n        with path.open(\'rb\') as handle:return hashlib.file_digest(handle,\'sha256\').hexdigest()==spec.sha256\n    progress(\'فحص أوزان النموذج\',force=True)\n    if valid(target): return target\n    if target.exists(): raise ValueError(\'Checkpoint checksum mismatch; cached model was not loaded\')\n    partial=target.with_suffix(target.suffix+\'.part\')\n    deadline=time.monotonic()+900\n    last_error=None\n    for attempt in range(3):\n        offset=partial.stat().st_size if partial.exists() else 0\n        if offset==spec.size and valid(partial): partial.replace(target);return target\n        if offset>=spec.size:partial.unlink();offset=0\n        progress(\'تنزيل أوزان النموذج\',offset,spec.size,force=True)\n        request=urllib.request.Request(spec.url,headers={\'Range\':f\'bytes={offset}-\'} if offset else {})\n        try:\n            with urllib.request.urlopen(request,timeout=20) as response:\n                status=response.getcode()\n                if status==206:\n                    if not response.headers.get(\'Content-Range\',\'\').startswith(f\'bytes {offset}-\'):raise ValueError(\'Invalid checkpoint byte range\')\n                elif offset:\n                    offset=0\n                with partial.open(\'ab\' if status==206 and offset else \'wb\') as out:\n                    downloaded=offset\n                    while True:\n                        if time.monotonic()>deadline:raise TimeoutError(\'Checkpoint download exceeded 15 minutes\')\n                        block=response.read(65536)\n                        if not block:break\n                        downloaded+=len(block)\n                        if downloaded>spec.size:raise ValueError(\'Checkpoint exceeds its declared size\')\n                        out.write(block);progress(\'تنزيل أوزان النموذج\',downloaded,spec.size)\n            progress(\'التحقق من بصمة النموذج\',force=True)\n            if not valid(partial):raise ValueError(\'Checkpoint is incomplete or its SHA256 differs\')\n            partial.replace(target);progress(\'اكتمل تنزيل النموذج\',spec.size,spec.size,force=True);return target\n        except (OSError,ValueError) as error:\n            last_error=error\n            if attempt<2 and time.monotonic()<deadline:\n                progress(\'إعادة اتصال بتنزيل النموذج؛ الجزء المحمّل محفوظ\',force=True)\n                continue\n            break\n    raise RuntimeError(\'تعذر تنزيل أوزان النموذج؛ أعد تشغيل الخلية لاستكمال التنزيل\') from last_error\n\n\ndef main(root, second):\n    global STATE\n    root=Path(root);STATE=root/\'bandit_progress.json\'\n    progress(\'تجهيز Torch وفحص GPU\',force=True)\n    import numpy as np\n    import soundfile as sf\n    import torch\n    from bandit_infer import BanditSession\n    from bandit_infer.checkpoints import get_spec\n    if not torch.cuda.is_available():raise RuntimeError(\'CUDA is required; CPU separation is disabled\')\n    audio,rate=sf.read(root/\'bandit_input.wav\',dtype=\'float32\',always_2d=True)\n    if rate!=48000 or audio.shape[1]!=2 or len(audio)==0:raise ValueError(\'Invalid test audio\')\n    spec=get_spec(\'v2-multi\')\n    started=time.monotonic()\n    weights=ensure_weights(root/\'bandit_weights\',spec)\n    progress(\'تحميل النموذج إلى GPU\',force=True)\n    with BanditSession(\'v2-multi\',family=\'v2\',device=\'cuda\',checkpoint_path=weights,checkpoint_sha256=spec.sha256) as model:\n        loading=time.monotonic()-started\n        runtime=getattr(model,\'_runtime\',None)\n        hook=None\n        count=[0];phase=[\'فصل الحوار والموسيقى والمؤثرات\']\n        total=runtime.handler._get_n_chunks(len(audio)) if runtime is not None else 0\n        if runtime is not None:\n            def completed(module,args,output):\n                torch.cuda.synchronize();count[0]+=1\n                progress(phase[0],count[0],total,force=True)\n            hook=runtime.model.register_forward_hook(completed)\n        def infer(values,label):\n            count[0]=0;phase[0]=label;progress(label,0,total,force=True)\n            torch.cuda.synchronize();began=time.monotonic()\n            values=model.infer(values,sample_rate=48000)\n            torch.cuda.synchronize()\n            return values,time.monotonic()-began\n        try:\n            stems,elapsed=infer(audio.T,\'فصل الحوار والموسيقى والمؤثرات\')\n            def mix(values,name):\n                if set(values)!={\'speech\',\'music\',\'effects\'}:raise ValueError(\'Unexpected stems\')\n                for value in values.values():\n                    if value.shape!=audio.T.shape or not np.isfinite(value).all():raise ValueError(\'Invalid stem shape/values\')\n                clean=values[\'speech\']+values[\'effects\'];peak=float(np.max(np.abs(clean)))\n                gain=min(1.0,.999/max(peak,1e-9))\n                sf.write(root/name,(clean*gain).T,48000,subtype=\'PCM_24\')\n                return clean,gain\n            progress(\'حفظ الحوار والمؤثرات\',force=True)\n            clean,gain=mix(stems,\'bandit_dialogue_fx.wav\')\n            sf.write(root/\'bandit_dialogue_only_reference.wav\',stems[\'speech\'].T,48000,subtype=\'FLOAT\')\n            second_seconds=None\n            if second:\n                again,second_seconds=infer(clean,\'الفصل الإضافي الاختياري\')\n                mix(again,\'bandit_second_pass.wav\')\n            info=model.cache_info()\n        finally:\n            if hook is not None:hook.remove()\n    report=dict(model=\'Bandit v2 Multi\',sample_rate=rate,frames=len(audio),duration=len(audio)/rate,\n        loading_seconds=loading,inference_seconds=elapsed,second_pass_seconds=second_seconds,\n        realtime_factor=(len(audio)/rate)/max(elapsed,1e-9),output_gain=gain,\n        torch=torch.__version__,gpu=torch.cuda.get_device_name(0),checkpoint=info)\n    (root/\'bandit_report.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2,default=str))\n    progress(\'اكتمل الفصل؛ شغّل خلية المعاينة والتنزيل\',force=True)\n\nif __name__==\'__main__\':\n    try:main(sys.argv[1],sys.argv[2]==\'1\')\n    except BaseException:\n        if STATE:progress(\'توقفت المرحلة؛ التفاصيل في سجل الخلية\',force=True)\n        raise\n',encoding="utf-8")
# Run inside the existing notebook kernel. The baseline/input remain in place.
import os, signal, time, json, subprocess
from pathlib import Path
RUNNER=ROOT/'bandit_runner.py'
# Only stop stale children running this exact job's old Bandit runner.
for entry in Path('/proc').iterdir():
    if not entry.name.isdigit() or int(entry.name)==os.getpid():continue
    try:
        args=(entry/'cmdline').read_bytes().split(b'\0')
        if str(RUNNER.resolve()).encode() not in args:continue
        os.kill(int(entry.name),signal.SIGTERM)
        until=time.monotonic()+5
        while entry.exists() and time.monotonic()<until:time.sleep(.1)
        if entry.exists():os.kill(int(entry.name),signal.SIGKILL)
    except (OSError,ValueError):pass
BANDIT_INPUT=RESULTS/'bandit_input.wav'
if not BANDIT_INPUT.exists():
    subprocess.run(['ffmpeg','-v','error','-y','-i',str(SEGMENT),'-vn','-ar','48000','-ac','2','-c:a','pcm_s16le',str(BANDIT_INPUT)],check=True)
STATE=RESULTS/'bandit_progress.json';STATE.unlink(missing_ok=True)
LOG=RESULTS/'bandit_worker.log'
started=time.monotonic();previous=None;last_print=0
with LOG.open('w') as log:
    process=subprocess.Popen([BPY,'-u',str(RUNNER),str(RESULTS),'1' if SECOND_PASS else '0'],stdout=log,stderr=log)
    try:
        while True:
            try:state=json.loads(STATE.read_text())
            except (OSError,ValueError):state={'stage':'بدء عامل الفصل'}
            marker=(state.get('stage'),state.get('completed'),state.get('total'))
            elapsed=time.monotonic()-started
            if marker!=previous or elapsed-last_print>=10:
                detail=state.get('stage','')
                if state.get('total',0):
                    done,total=state['completed'],state['total']
                    if total>1000000:detail+=f' · {done/1e6:.1f} / {total/1e6:.1f} MB'
                    else:detail+=f' · الجزء {done} / {total}'
                print(f'{elapsed:.0f} ثانية · {detail}',flush=True)
                previous=marker;last_print=elapsed
            code=process.poll()
            if code is not None:
                if code:raise RuntimeError(LOG.read_text(errors='replace')[-6000:])
                break
            if elapsed>1800:raise TimeoutError('انتهت مهلة التجربة؛ السجل محفوظ ويمكن استكمال تنزيل الأوزان')
            time.sleep(2)
    finally:
        if process.poll() is None:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
print('اكتمل الفصل. شغّل الخلية 7 للمعاينة والتنزيل.',flush=True)


In [ ]:
#@title 7. الاستماع والمقارنة وتنزيل الفيديو أو الصوت
from IPython.display import display, Audio, Video, Markdown
import zipfile
CANDIDATE=json.loads((RESULTS/'bandit_report.json').read_text())
CANDIDATE['runtime_revision']=BANDIT_REV
CANDIDATE['baseline_model_fingerprint']=BASELINE_RESULT['model_sha256']
CANDIDATE['baseline_inference_seconds']=BASELINE_SECONDS
CANDIDATE['selected_start_seconds']=START_SECONDS
(RESULTS/'comparison_report.json').write_text(json.dumps(CANDIDATE,ensure_ascii=False,indent=2),encoding='utf-8')
variants=[('الأصل',BANDIT_INPUT),('الحالي: HQ4 ثم HQ3',FINAL_WAV),('المرشح: الحوار والمؤثرات',RESULTS/'bandit_dialogue_fx.wav')]
if SECOND_PASS: variants.append(('المرشح: مرحلة إضافية تجريبية',RESULTS/'bandit_second_pass.wav'))
for label,path in variants:
    display(Markdown('### '+label))
    # normalize=False: do not hide quiet dialogue or residual music with per-clip normalization.
    display(Audio(filename=str(path),normalize=False))
    if HAS_VIDEO and path!=BANDIT_INPUT:
        target=RESULTS/(path.stem+'.mp4')
        subprocess.run(['ffmpeg','-v','error','-y','-i',str(SEGMENT),'-i',str(path),
          '-map','0:v:0','-map','1:a:0','-c:v','copy','-c:a','aac','-b:a','192k',
          '-movflags','+faststart','-shortest',str(target)],check=True)
        display(Video(str(target),embed=True,width=480))
print('زمن الفصل الحالي:',round(BASELINE_SECONDS,2),'ثانية')
print('زمن فصل Bandit:',round(CANDIDATE['inference_seconds'],2),'ثانية، التهيئة منفصلة:',round(CANDIDATE['loading_seconds'],2))
print('للتقييم: بقايا الموسيقى، وضوح الكلمات، بقاء الخطوات والرياح والمؤثرات، وعدم تقطع الصوت. لا يوجد قياس دقة موثوق دون مسارات أصلية منفصلة.')
print('الملف dialogue_only_reference للمقارنة فقط؛ لا يحافظ على المؤثرات.')
ARCHIVE=ROOT/'Madar-Separation-Comparison.zip'
with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_STORED) as archive:
    selected=[FINAL_WAV,RESULTS/'bandit_dialogue_fx.wav',RESULTS/'bandit_dialogue_only_reference.wav',RESULTS/'comparison_report.json']
    selected+=list(RESULTS.glob('*.mp4'))
    if SECOND_PASS: selected.append(RESULTS/'bandit_second_pass.wav')
    for path in selected: archive.write(path,path.name)
files.download(str(ARCHIVE))


## بعد المقارنة

ابدأ بجزء قصير يحتوي على المشكلة، ثم جرّب حوارًا هادئًا ومشهد حركة وموسيقى مرتفعة.
أرسل اسم النتيجة الأفضل، وهل اختفت أصوات محيطية أو حروف من الحوار.
لا يعني نجاح نموذج في مقطع أنه أدق في جميع المقاطع. المرحلة الإضافية قد تزيد التنظيف وتزيد خسارة التفاصيل معًا.
إذا نفدت ذاكرة GPU، قلل مدة التجربة؛ لا يغيّر الدفتر بطاقتك ولا يتحول تلقائيًا إلى CPU.
